<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/5-model-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CNN VS FINE-TUNING VS RANDOM FOREST


Implementando Pytorch/Torchvision, podemos comparar o desempenño de um modelo CNN pré-treinado, un modelo fine-tuning y unmodelo Random Forest para la tarea de clasificacion de imagenes.

El imput para probar nuestros modelos sera un archivo de audio nuevo y grabado por nosotros mismos, este clip primero sera filtrado, recortado y convertido a un espectrograma, los parametros de dicha transformación deberan ajustarse al formato de entrada al que hemos configurado como nuevos inputs para nuestros modelos.

Para ello sera destinada una web app, donde el usuario pueda subir su clip de audio, y el sistema se encargara de procesarlo y mostrar los resultados de las predicciones de cada modelo.

In [27]:
# importando librerias necesarias

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import GradScaler, autocast # Para eficiencia en memoria
from google.colab import drive


In [28]:
# Preprocesamiento y obtencion de conjuntos para entrenamiento, test, validacion.
#--------------------------------------------------------------------------------

# Conectar a Google Drive

drive.mount('/content/drive')
ROOT_DIR = '/content/drive/MyDrive/ravdess_images_02'  # Cambia esto a tu ruta

# Configuración del dispositivo (GPU si está disponible, de lo contrario CPU)
# Se recomienda usar la GPU que ofrece Google Colab para acelerar el entrenamiento de

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Estas en modo: {device}")

# ImageFolder asume automaticamente que dentro de la carpeta en roo_dir estan las clases
# En nuestro caso, son las emociones.

def get_dataloaders(base_dir, feature_type,  batch_size=32):

  # Preprocessing: Resize for ResNet, conversio a tensores, normalizar (ImageNet stats )
  data_transforms = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalización para canales RGB necesario en ResNet
  ])

  # Define paths
  # Assuming your structure is: ./dataset/features/mfcc/calm/01.png

  TARGET_FEATURE = feature_type # Change to 'spec', 'melspect', 'delta', etc.
  FEATURE_DIR = os.path.join(ROOT_DIR, TARGET_FEATURE)

  # Use PyTorch's native ImageFolder!
  full_dataset = datasets.ImageFolder(root=FEATURE_DIR, transform=data_transforms)

  # The class map is automatically generated from folder names
  class_names = full_dataset.classes
  print(f"Emociones detectadas: {class_names}")
  print(f"Total images para la clase {TARGET_FEATURE}: {len(full_dataset)}")

  # 80-10-10 Split for Train, Validation, Test
  train_size = int(0.8 * len(full_dataset))
  val_size = int(0.1 * len(full_dataset))
  test_size = len(full_dataset) - train_size - val_size

  train_dataset, val_dataset, test_dataset = random_split(
      full_dataset, [train_size, val_size, test_size],
      generator=torch.Generator().manual_seed(42)
  )

  # Create DataLoaders
  batch_size = 32
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

  print("\n")
  print("El tamaño de los sets es:")
  print(f"Entrenamiento: {len(train_dataset)}")
  print(f"Validacion: {len(val_dataset)}")
  print(f"Test: {len(test_dataset)}")

  return train_loader, val_loader, test_loader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Estas en modo: cuda


In [8]:
# Print root_dir content:
print(os.listdir(ROOT_DIR))

['mfcc', 'delta', 'delta2', 'spec', 'mel_spec']


In [15]:
resnet_feature_selector(ROOT_DIR, 'mel_spec')

Emociones detectadas: ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Total images para la clase mel_spec: 1433


El tamaño de los sets es:
Entrenamiento: 1146
Validacion: 143
Test: 144


In [17]:
# 3. CONSTRUCCIÓN DEL MODELO (FINE-TUNING)
# -------------------------------------------------------------------------------

def build_model(num_classes):
    """
    Carga ResNet18 preentrenado, congela las primeras capas y adapta la capa final.
    """
    # Cargamos los pesos preentrenados de ImageNet
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Congelamos los parámetros de las primeras capas,
    # ya que esas capas ya saben detectar bordes y colores básicos.
    for param in model.parameters():
        param.requires_grad = False

    # Descongelamos el último bloque convolucional (layer4) para aprender
    # características específicas de nuestros espectrogramas/MFCCs
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Reemplazamos la última capa completamente conectada (fc) para que
    # coincida con nuestro número de emociones (ej. 8 en RAVDESS)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model.to(device)

In [20]:
from functools import total_ordering
# Entrenamiento mediante cliclo optimizado (Aqui es donde realmente usamos la GPU)

def train_model(model, train_loader, val_loader, feature_type, epochs=15, lr=1e-4):  # Learnig Rate de 0.0001

  """
  Bucle de entrenamiento, usamos AMP (Automatic Mixed Precision) para acelerar el proceso usando 16bits
  de esa manera se consume menos VRAM

  """

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

  # Para evitar el subdesbordamiento (uderflow) en APM.
  scaler = GradScaler()
  best_val_loss = float('inf')
  best_model_path = f'best_model_{feature_type}.pth'

  print(f"Iniciando entrenamiento para la feature: {feature_type}")

  for epoch in range(epochs):
      model.train()
      running_loss, correct, total = 0.0, 0 , 0

      for inputs, labels in train_loader:
          inputs, labels = inputs.to(device), labels.to(device)
          optimizer.zero_grad()
          # Para castear dinámicamente tensores a FP16
          with autocast():
              outputs = model(inputs)
              loss = criterion(outputs, labels)
          # Backward pass escalado para mantener precisión
          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()

          # Cálculos de métricas de entrenamiento
          running_loss += loss.item() * inputs.size(0)
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

      train_loss = running_loss / total
      train_acc = correct / total

      # Validacion

      model.eval()
      val_loss, val_correct, val_total = 0.0, 0, 0

      # Desactivamos gradientes para ahorrar memoria y tiempo en validación
      with torch.no_grad():

        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

      val_loss = val_loss / val_total
      val_acc = val_correct / val_total

      print(f"Época {epoch+1}/{epochs} | Loss Entr: {train_loss:.4f} Acc Entr: {train_acc:.4f} | Loss Val: {val_loss:.4f} Acc Val: {val_acc:.4f}")

      # Guardado de checkpoints del modelo (solo si mejora la pérdida de validación)
      if val_loss < best_val_loss:
          best_val_loss = val_loss
          torch.save(model.state_dict(), best_model_path)

  print(f"Entrenamiento completado. Mejor modelo guardado como '{best_model_path}'\n")
  return best_model_path

In [26]:

TARGET_FEATURE = 'mfcc' # Aquí defines si es 'mfcc', 'spec', 'delta', etc.

# Cargamos los datos
train_loader, val_loader, test_loader, class_names = resnet_feature_selector(
    base_dir=ROOT_DIR,
    feature_type=TARGET_FEATURE,
    batch_size=32
)

TypeError: resnet_feature_selector() got an unexpected keyword argument 'base_dir'